# 05 — KPIs: Indicadores Clave de Desempeño — Chicago Crimes COVID-19 (Cloud — GCS)

Versión cloud del notebook KPIs. Los tres frameworks leen datos desde **GCS** (`gs://big-data-proyecto-parcial/raw/`) en lugar del disco local.

| # | KPI | Framework | Categoría |
|---|-----|-----------|--------------------|
| 1 | Variación de la tasa de arresto global por era | Dask | Efectividad policial |
| 2 | Caída de criminalidad en el pico del confinamiento (2020) | Dask | Tendencia COVID |
| 3 | Aumento de violencia doméstica durante el COVID | Modin | Violencia doméstica |
| 4 | Tipo de crimen con mayor cambio de resolución entre eras | Modin | Efectividad policial |
| 5 | Redistribución geográfica: distritos que ganaron/perdieron crimen | Modin | Geografía |
| 6 | Recuperación post-COVID: comparativa POST vs PRE | Dask | Tendencia COVID |
| 7 | Cambio en el índice de criminalidad nocturna por era | Dask | Temporalidad |
| 8 | Variación de crímenes violentos (FBI Part I) por era | Spark | Efectividad policial |
| 9 | Diversidad de crímenes por era — Índice de Shannon | Spark | Geografía |
| 10 | Variación en calidad del registro CPD durante el COVID (MTTR) | Spark | Calidad de datos |

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'gcsfs', 'modin[ray]', 'pandas-gbq', '--quiet'], check=True)
print('Dependencias cloud OK')

Dependencias cloud OK


In [2]:
import dask.dataframe as dd
import modin.pandas as mpd
import pandas as pd
import numpy as np
import os
import ray
import warnings
warnings.filterwarnings('ignore')

PROJECT_ID     = 'my-first-project-492901'
DATASET_ID     = 'chicago_crimes_results'
GCS_BUCKET     = 'gs://big-data-proyecto-parcial/raw'
YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP        = {2017:'PRE', 2018:'PRE', 2019:'PRE',
                  2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
                  2023:'POST', 2024:'POST', 2025:'POST'}

files       = [f'{GCS_BUCKET}/Chicago_Crimes_{y}.csv' for y in YEARS_ANALYSIS]
GCS_STORAGE = {"token": "google_default"}

def to_bigquery(df: pd.DataFrame, table_name: str, if_exists: str = 'replace') -> None:
    df.to_gbq(
        destination_table=f'{DATASET_ID}.{table_name}',
        project_id=PROJECT_ID,
        if_exists=if_exists,
        progress_bar=False,
    )
    print(f'  → BigQuery: {PROJECT_ID}.{DATASET_ID}.{table_name}  ({len(df):,} filas)')

kpis = []
print(f'Fuente: GCS — {GCS_BUCKET}/')
print('Entorno listo. Período: 2017–2025 (PRE / DURANTE / POST COVID)')

Fuente: GCS — gs://big-data-proyecto-parcial/raw/
Entorno listo. Período: 2017–2025 (PRE / DURANTE / POST COVID)


In [3]:
# ── Carga con Dask desde GCS (KPIs 1, 2, 6, 7) ───────────────────────────────
dtypes = {
    'unique_key': 'Int64', 'case_number': 'object', 'block': 'object',
    'iucr': 'object', 'primary_type': 'object', 'description': 'object',
    'location_description': 'object', 'beat': 'Int64', 'district': 'Int64',
    'ward': 'Int64', 'community_area': 'Int64', 'fbi_code': 'object',
    'x_coordinate': 'Int64', 'y_coordinate': 'Int64', 'year': 'Int64',
    'latitude': 'float64', 'longitude': 'float64', 'location': 'object',
}
ddf = dd.read_csv(
    files, dtype=dtypes, parse_dates=['date', 'updated_on'],
    assume_missing=True, storage_options=GCS_STORAGE,
)
ddf['covid_era'] = ddf['year'].map(ERA_MAP, meta=('covid_era', 'object'))
total = len(ddf)
print(f'Total registros (Dask, GCS 2017–2025): {total:,}')

Total registros (Dask, GCS 2017–2025): 2,072,943


## KPI 1 — Variación de la Tasa de Arresto por Era COVID *(Dask — GCS)*

In [4]:
arrest_by_era = (
    ddf.groupby('covid_era')
       .agg({'arrest': 'sum', 'unique_key': 'count'})
       .compute()
       .rename(columns={'arrest': 'arrests', 'unique_key': 'total'})
       .reset_index()
)
arrest_by_era['arrest_rate_pct'] = (arrest_by_era['arrests'] / arrest_by_era['total'] * 100).round(2)

pre_rate     = float(arrest_by_era.loc[arrest_by_era['covid_era']=='PRE',    'arrest_rate_pct'].values[0])
durante_rate = float(arrest_by_era.loc[arrest_by_era['covid_era']=='DURANTE','arrest_rate_pct'].values[0])
post_rate    = float(arrest_by_era.loc[arrest_by_era['covid_era']=='POST',   'arrest_rate_pct'].values[0])
delta_kpi1   = round(durante_rate - pre_rate, 2)

print('KPI 1 | Variación de Tasa de Arresto por Era COVID')
for _, row in arrest_by_era.sort_values('covid_era').iterrows():
    print(f"  {row['covid_era']:<8}: {row['arrest_rate_pct']:.2f}%  ({int(row['total']):,} crímenes)")
print(f"  ΔDURANTE-PRE : {delta_kpi1:+.2f} pp")

kpis.append({'kpi_id': 1, 'kpi_name': 'Variación Tasa de Arresto PRE→DURANTE',
             'value_numeric': delta_kpi1,
             'value_text': f'PRE:{pre_rate:.2f}% → DURANTE:{durante_rate:.2f}% → POST:{post_rate:.2f}%',
             'unit': 'pp', 'framework': 'Dask'})

KPI 1 | Variación de Tasa de Arresto por Era COVID
  DURANTE : 13.42%  (661,583 crímenes)
  POST    : 13.32%  (611,521 crímenes)
  PRE     : 20.36%  (799,839 crímenes)
  ΔDURANTE-PRE : -6.94 pp


## KPI 2 — Caída de Criminalidad en el Pico del Confinamiento (2020) *(Dask — GCS)*

In [5]:
crimes_by_year = (
    ddf.groupby('year')['unique_key']
       .count().compute()
       .reset_index()
       .rename(columns={'unique_key': 'total'})
       .sort_values('year')
)
crimes_by_year['covid_era'] = crimes_by_year['year'].map(ERA_MAP)
crimes_by_year['yoy_pct']   = crimes_by_year['total'].pct_change() * 100

n2019 = int(crimes_by_year.loc[crimes_by_year['year']==2019, 'total'].values[0])
n2020 = int(crimes_by_year.loc[crimes_by_year['year']==2020, 'total'].values[0])
kpi2  = round((n2020 - n2019) / n2019 * 100, 2)

print('KPI 2 | Caída de Criminalidad en el Pico del Confinamiento (2020)')
print(f'  Crímenes 2019 : {n2019:,}')
print(f'  Crímenes 2020 : {n2020:,}')
print(f'  Variación     : {kpi2:+.2f}%')
print('\nEvolución año a año:')
print(crimes_by_year[['year','covid_era','total','yoy_pct']].to_string(index=False))

kpis.append({'kpi_id': 2, 'kpi_name': 'Caída de Criminalidad en Confinamiento 2020',
             'value_numeric': kpi2, 'value_text': f'{kpi2:+.2f}% (2019→2020)',
             'unit': '%', 'framework': 'Dask'})

KPI 2 | Caída de Criminalidad en el Pico del Confinamiento (2020)
  Crímenes 2019 : 261,555
  Crímenes 2020 : 212,522
  Variación     : -18.75%

Evolución año a año:
 year covid_era  total    yoy_pct
 2017       PRE 269214       <NA>
 2018       PRE 269070  -0.053489
 2019       PRE 261555  -2.792954
 2020   DURANTE 212522 -18.746726
 2021   DURANTE 209406  -1.466201
 2022   DURANTE 239655  14.445145
 2023      POST 262756   9.639273
 2024      POST 256305  -2.455129
 2025      POST  92460 -63.925792


## KPI 3–5 con Modin desde GCS

In [6]:
ray.init(ignore_reinit_error=True)
# Leer desde GCS via gcsfs
mdf = mpd.concat(
    [mpd.read_csv(f, low_memory=False, storage_options=GCS_STORAGE) for f in files],
    ignore_index=True,
)
mdf['covid_era']     = mdf['year'].map(ERA_MAP)
mdf['domestic_bool'] = mdf['domestic'].astype(str).str.lower().eq('true')
mdf['arrest_bool']   = mdf['arrest'].astype(str).str.lower().eq('true')
print(f'Modin cargado desde GCS: {mdf.shape[0]:,} filas')

2026-05-02 02:25:28,665	INFO worker.py:2012 -- Started a local Ray instance.


Modin cargado desde GCS: 2,072,943 filas


## KPI 3 — Aumento de Violencia Doméstica *(Modin — GCS)*

In [7]:
dom_by_era = (
    mdf.groupby('covid_era')
       .agg(domestic_crimes=('domestic_bool', 'sum'), total=('unique_key', 'count'))
       .reset_index()
)
dom_by_era['domestic_rate_pct'] = (dom_by_era['domestic_crimes'] / dom_by_era['total'] * 100).round(2)

pre_dom     = float(dom_by_era.loc[dom_by_era['covid_era']=='PRE',    'domestic_rate_pct'].values[0])
durante_dom = float(dom_by_era.loc[dom_by_era['covid_era']=='DURANTE','domestic_rate_pct'].values[0])
post_dom    = float(dom_by_era.loc[dom_by_era['covid_era']=='POST',   'domestic_rate_pct'].values[0])
delta_kpi3  = round(durante_dom - pre_dom, 2)

print('KPI 3 | Aumento de Violencia Doméstica Durante el COVID')
for _, row in dom_by_era.sort_values('covid_era').iterrows():
    print(f"  {row['covid_era']:<8}: {row['domestic_rate_pct']:.2f}%  ({int(row['domestic_crimes']):,} de {int(row['total']):,})")
print(f"  ΔDURANTE-PRE : {delta_kpi3:+.2f} pp")

kpis.append({'kpi_id': 3, 'kpi_name': 'Aumento de Violencia Doméstica DURANTE COVID',
             'value_numeric': delta_kpi3,
             'value_text': f'PRE:{pre_dom:.2f}% → DURANTE:{durante_dom:.2f}% → POST:{post_dom:.2f}%',
             'unit': 'pp', 'framework': 'Modin'})

KPI 3 | Aumento de Violencia Doméstica Durante el COVID
  DURANTE : 21.02%  (139,095 de 661,583)
  POST    : 18.39%  (112,471 de 611,521)
  PRE     : 19.20%  (153,534 de 799,839)
  ΔDURANTE-PRE : +1.82 pp


## KPI 4 — Tipo con Mayor Cambio de Resolución *(Modin — GCS)*

In [8]:
rate_by_type_era = (
    mdf.groupby(['primary_type', 'covid_era'])
       .agg(arrests=('arrest_bool', 'sum'), total=('unique_key', 'count'))
       .reset_index()
)
rate_by_type_era['rate'] = (rate_by_type_era['arrests'] / rate_by_type_era['total'] * 100).round(2)

pivot_rate = rate_by_type_era.pivot(index='primary_type', columns='covid_era', values='rate').dropna()
pivot_rate['delta_durante_pre'] = (pivot_rate['DURANTE'] - pivot_rate['PRE']).round(2)
pivot_rate['abs_delta']         = pivot_rate['delta_durante_pre'].abs()
pivot_rate = pivot_rate.sort_values('abs_delta', ascending=False)

top_change = pivot_rate.iloc[0]
kpi4_type  = pivot_rate.index[0]
kpi4_delta = float(top_change['delta_durante_pre'])

print('KPI 4 | Tipo con Mayor Cambio de Resolución PRE→DURANTE')
print(f'  Tipo    : {kpi4_type}')
print(f'  PRE     : {top_change["PRE"]:.2f}%')
print(f'  DURANTE : {top_change["DURANTE"]:.2f}%')
print(f'  Δ       : {kpi4_delta:+.2f} pp')

kpis.append({'kpi_id': 4, 'kpi_name': 'Tipo con Mayor Cambio de Resolución por COVID',
             'value_numeric': kpi4_delta,
             'value_text': f'{kpi4_type}: {kpi4_delta:+.2f} pp (PRE→DURANTE)',
             'unit': 'pp', 'framework': 'Modin'})

KPI 4 | Tipo con Mayor Cambio de Resolución PRE→DURANTE
  Tipo    : PUBLIC PEACE VIOLATION
  PRE     : 66.49%
  DURANTE : 38.61%
  Δ       : -27.88 pp


## KPI 5 — Redistribución Geográfica *(Modin — GCS)*

In [9]:
dist_by_era = (
    mdf.dropna(subset=['district'])
       .groupby(['district', 'covid_era'])
       .size().reset_index(name='count')
)
dist_pivot = dist_by_era.pivot(index='district', columns='covid_era', values='count').dropna()
dist_pivot['annual_pre']     = dist_pivot['PRE']     / 3
dist_pivot['annual_durante'] = dist_pivot['DURANTE'] / 3
dist_pivot['delta_pct']      = ((dist_pivot['annual_durante'] - dist_pivot['annual_pre']) /
                                 dist_pivot['annual_pre'] * 100).round(2)
dist_pivot = dist_pivot.sort_values('delta_pct')

top_fall_dist = int(dist_pivot.index[0])
kpi5_delta    = float(dist_pivot.iloc[0]['delta_pct'])

print('KPI 5 | Redistribución Geográfica por COVID')
print(f'  Distrito con mayor caída : Distrito {top_fall_dist}  ({kpi5_delta:+.2f}%)')
print(f'  Distrito con menor caída : Distrito {int(dist_pivot.index[-1])}  ({dist_pivot.iloc[-1]["delta_pct"]:+.2f}%)')

kpis.append({'kpi_id': 5, 'kpi_name': 'Distrito con Mayor Caída de Crimen en COVID',
             'value_numeric': kpi5_delta,
             'value_text': f'Distrito {top_fall_dist}: {kpi5_delta:+.2f}% (DURANTE vs PRE anualizado)',
             'unit': '%', 'framework': 'Modin'})

ray.shutdown()

KPI 5 | Redistribución Geográfica por COVID
  Distrito con mayor caída : Distrito 1  (-33.89%)
  Distrito con menor caída : Distrito 31  (+66.67%)


## KPI 6 y 7 — Recuperación y Criminalidad Nocturna *(Dask — GCS)*

In [10]:
# KPI 6 — Recuperación post-COVID vs nivel PRE
avg_pre  = float(crimes_by_year[crimes_by_year['covid_era']=='PRE']['total'].mean())
avg_post = float(crimes_by_year[crimes_by_year['covid_era']=='POST']['total'].mean())
kpi6     = round((avg_post - avg_pre) / avg_pre * 100, 2)

print('KPI 6 | Recuperación Post-COVID vs Nivel PRE')
print(f'  Promedio anual PRE  (2017–2019): {avg_pre:,.0f}')
print(f'  Promedio anual POST (2023–2025): {avg_post:,.0f}')
print(f'  Recuperación: {kpi6:+.2f}%')

kpis.append({'kpi_id': 6, 'kpi_name': 'Recuperación Post-COVID vs Nivel PRE',
             'value_numeric': kpi6, 'value_text': f'{kpi6:+.2f}% (POST vs PRE anualizado)',
             'unit': '%', 'framework': 'Dask'})

# KPI 7 — Criminalidad nocturna
ddf['hour']     = ddf['date'].dt.hour
ddf['is_night'] = (ddf['hour'] >= 22) | (ddf['hour'] <= 5)

night_by_era = (
    ddf.groupby('covid_era')
       .agg({'is_night': 'sum', 'unique_key': 'count'})
       .compute()
       .rename(columns={'is_night': 'night_crimes', 'unique_key': 'total'})
       .reset_index()
)
night_by_era['night_rate_pct'] = (night_by_era['night_crimes'] / night_by_era['total'] * 100).round(2)

pre_night     = float(night_by_era.loc[night_by_era['covid_era']=='PRE',    'night_rate_pct'].values[0])
durante_night = float(night_by_era.loc[night_by_era['covid_era']=='DURANTE','night_rate_pct'].values[0])
post_night    = float(night_by_era.loc[night_by_era['covid_era']=='POST',   'night_rate_pct'].values[0])
kpi7          = round(durante_night - pre_night, 2)

print('\nKPI 7 | Cambio en Índice de Criminalidad Nocturna por Era')
for _, row in night_by_era.sort_values('covid_era').iterrows():
    print(f"  {row['covid_era']:<8}: {row['night_rate_pct']:.2f}%")
print(f"  ΔDURANTE-PRE : {kpi7:+.2f} pp")

kpis.append({'kpi_id': 7, 'kpi_name': 'Cambio en Índice de Criminalidad Nocturna',
             'value_numeric': kpi7,
             'value_text': f'PRE:{pre_night:.2f}% → DURANTE:{durante_night:.2f}% → POST:{post_night:.2f}%',
             'unit': 'pp', 'framework': 'Dask'})

KPI 6 | Recuperación Post-COVID vs Nivel PRE
  Promedio anual PRE  (2017–2019): 266,613
  Promedio anual POST (2023–2025): 203,840
  Recuperación: -23.54%



KPI 7 | Cambio en Índice de Criminalidad Nocturna por Era
  DURANTE : 28.01%
  POST    : 27.99%
  PRE     : 24.42%
  ΔDURANTE-PRE : +3.59 pp


## KPI 8–10 con Spark desde GCS (YARN en Dataproc, local[4] en PC)

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

YARN_MASTER = os.environ.get('YARN_CONF_DIR') is not None
spark_master = 'yarn' if YARN_MASTER else 'local[4]'

builder = (
    SparkSession.builder
    .appName('ChicagoCrimes-KPIs-COVID-Cloud')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
)
if YARN_MASTER:
    builder = builder.config('spark.executor.memory', '6g').config('spark.executor.cores', '2')

spark = builder.getOrCreate()

era_expr = F.when(F.col('year') <= 2019, 'PRE') \
            .when(F.col('year') <= 2022, 'DURANTE') \
            .otherwise('POST')

if YARN_MASTER:
    sdf_raw = (
        spark.read
             .option('header', 'true')
             .option('nullValue', '')
             .option('inferSchema', 'true')
             .csv(files)    # gs:// URIs — GCS connector nativo en Dataproc
    )
else:
    # PC local: gcsfs/pandas → Spark
    import gcsfs as _gcsfs
    import pandas as _pd_spark
    print('Modo local: leyendo desde GCS via gcsfs para Spark...')
    _fs = _gcsfs.GCSFileSystem(token='google_default')
    _bucket = GCS_BUCKET.replace('gs://', '')
    _dfs = []
    for _y in YEARS_ANALYSIS:
        with _fs.open(f'{_bucket}/Chicago_Crimes_{_y}.csv', 'rb') as _f:
            _dfs.append(_pd_spark.read_csv(_f, dtype=str, low_memory=False))
    _pdf = _pd_spark.concat(_dfs, ignore_index=True)
    for _c in ['unique_key','beat','district','ward','community_area','x_coordinate','y_coordinate','year']:
        _pdf[_c] = _pd_spark.to_numeric(_pdf[_c], errors='coerce')
    for _c in ['latitude','longitude']:
        _pdf[_c] = _pd_spark.to_numeric(_pdf[_c], errors='coerce')
    sdf_raw = spark.createDataFrame(_pdf)
    print(f'  Spark DataFrame creado: {sdf_raw.count():,} registros')

sdf = (
    sdf_raw
    .withColumn('date',      F.to_timestamp('date'))
    .withColumn('covid_era', era_expr)
    .filter(F.col('year').between(2017, 2025))
)
sdf.createOrReplaceTempView('crimes')
print(f'Spark {spark.version} — modo {spark_master}, fuente: GCS')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/02 02:26:31 INFO SparkEnv: Registering MapOutputTracker
26/05/02 02:26:31 INFO SparkEnv: Registering BlockManagerMaster
26/05/02 02:26:31 INFO SparkEnv: Registering BlockManagerMasterHeartbeat


26/05/02 02:26:31 INFO SparkEnv: Registering OutputCommitCoordinator


Modo local: leyendo desde GCS via gcsfs para Spark...


26/05/02 02:29:20 WARN TaskSetManager: Stage 0 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


  Spark DataFrame creado: 2,072,943 registros
Spark 3.5.3 — modo local[4], fuente: GCS


## KPI 8 — Variación Crímenes Violentos FBI Part I *(Spark — GCS)*

In [12]:
violent_by_era = spark.sql("""
    SELECT
        covid_era,
        COUNT(*) AS total,
        SUM(CASE WHEN fbi_code IN ('01A','01B','02','03','04A','04B','05','06','07','09')
                 THEN 1 ELSE 0 END) AS violent,
        ROUND(
            SUM(CASE WHEN fbi_code IN ('01A','01B','02','03','04A','04B','05','06','07','09')
                     THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        ) AS violent_rate_pct
    FROM crimes
    WHERE covid_era IS NOT NULL
    GROUP BY covid_era
    ORDER BY covid_era
""")

violent_pdf = violent_by_era.toPandas()
pre_vr     = float(violent_pdf.loc[violent_pdf['covid_era']=='PRE',    'violent_rate_pct'].values[0])
durante_vr = float(violent_pdf.loc[violent_pdf['covid_era']=='DURANTE','violent_rate_pct'].values[0])
post_vr    = float(violent_pdf.loc[violent_pdf['covid_era']=='POST',   'violent_rate_pct'].values[0])
kpi8       = round(durante_vr - pre_vr, 2)

print('KPI 8 | Variación de Crímenes Violentos (FBI Part I) por Era')
print(violent_pdf[['covid_era','total','violent','violent_rate_pct']].to_string(index=False))
print(f'\n  ΔDURANTE-PRE : {kpi8:+.2f} pp')

kpis.append({'kpi_id': 8, 'kpi_name': 'Variación Crímenes Violentos FBI Part I por COVID',
             'value_numeric': kpi8,
             'value_text': f'PRE:{pre_vr:.2f}% → DURANTE:{durante_vr:.2f}% → POST:{post_vr:.2f}%',
             'unit': 'pp', 'framework': 'Spark'})

26/05/02 02:29:28 WARN TaskSetManager: Stage 3 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


KPI 8 | Variación de Crímenes Violentos (FBI Part I) por Era
covid_era  total  violent violent_rate_pct
  DURANTE 661583   117144            17.71
     POST 611521   122668            20.06
      PRE 799839   121517            15.19

  ΔDURANTE-PRE : +2.52 pp


## KPI 9 — Diversidad de Crímenes (Shannon) *(Spark — GCS)*

In [13]:
proportions_era = spark.sql("""
    SELECT covid_era, primary_type,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY covid_era) AS proportion
    FROM crimes
    WHERE covid_era IS NOT NULL AND primary_type IS NOT NULL
    GROUP BY covid_era, primary_type
""")

shannon_era = (
    proportions_era
    .withColumn('contrib', -F.col('proportion') * F.log2(F.col('proportion')))
    .groupBy('covid_era')
    .agg(F.round(F.sum('contrib'), 4).alias('shannon_index'))
    .orderBy('covid_era')
)

shannon_pdf = shannon_era.toPandas()
pre_h     = float(shannon_pdf.loc[shannon_pdf['covid_era']=='PRE',    'shannon_index'].values[0])
durante_h = float(shannon_pdf.loc[shannon_pdf['covid_era']=='DURANTE','shannon_index'].values[0])
post_h    = float(shannon_pdf.loc[shannon_pdf['covid_era']=='POST',   'shannon_index'].values[0])
kpi9      = round(durante_h - pre_h, 4)

print('KPI 9 | Diversidad de Crímenes por Era (Índice de Shannon)')
print(shannon_pdf.to_string(index=False))
print(f'\n  ΔDURANTE-PRE : {kpi9:+.4f} bits')

kpis.append({'kpi_id': 9, 'kpi_name': 'Cambio en Diversidad de Crímenes (Shannon) por COVID',
             'value_numeric': kpi9,
             'value_text': f'PRE:{pre_h:.4f} → DURANTE:{durante_h:.4f} → POST:{post_h:.4f} bits',
             'unit': 'bits', 'framework': 'Spark'})

26/05/02 02:29:35 WARN TaskSetManager: Stage 11 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


KPI 9 | Diversidad de Crímenes por Era (Índice de Shannon)
covid_era  shannon_index
  DURANTE         3.4674
     POST         3.4228
      PRE         3.4469

  ΔDURANTE-PRE : +0.0205 bits


## KPI 10 — Variación MTTR Registro CPD *(Spark — GCS)*

In [14]:
mttr_by_era = spark.sql("""
    SELECT
        covid_era,
        ROUND(AVG(DATEDIFF(
            TO_DATE(CAST(updated_on AS STRING)),
            TO_DATE(CAST(date AS STRING))
        )), 1) AS avg_days,
        MIN(DATEDIFF(
            TO_DATE(CAST(updated_on AS STRING)),
            TO_DATE(CAST(date AS STRING))
        )) AS min_days
    FROM crimes
    WHERE date IS NOT NULL AND updated_on IS NOT NULL AND covid_era IS NOT NULL
    GROUP BY covid_era
    ORDER BY covid_era
""")

mttr_pdf     = mttr_by_era.toPandas()
pre_mttr     = float(mttr_pdf.loc[mttr_pdf['covid_era']=='PRE',    'avg_days'].values[0])
durante_mttr = float(mttr_pdf.loc[mttr_pdf['covid_era']=='DURANTE','avg_days'].values[0])
post_mttr    = float(mttr_pdf.loc[mttr_pdf['covid_era']=='POST',   'avg_days'].values[0])
kpi10        = round(durante_mttr - pre_mttr, 1)

print('KPI 10 | Variación en Calidad del Registro CPD (MTTR) por Era')
print(mttr_pdf[['covid_era','avg_days','min_days']].to_string(index=False))
print(f'\n  ΔDURANTE-PRE : {kpi10:+.1f} días')

kpis.append({'kpi_id': 10, 'kpi_name': 'Variación MTTR Registro CPD por COVID',
             'value_numeric': kpi10,
             'value_text': f'PRE:{pre_mttr:.1f}d → DURANTE:{durante_mttr:.1f}d → POST:{post_mttr:.1f}d',
             'unit': 'días', 'framework': 'Spark'})

26/05/02 02:29:41 WARN TaskSetManager: Stage 24 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


KPI 10 | Variación en Calidad del Registro CPD (MTTR) por Era
covid_era  avg_days  min_days
  DURANTE      87.8         3
     POST     112.0        -3
      PRE     125.8         6

  ΔDURANTE-PRE : -38.0 días


## Resumen ejecutivo y upload a BigQuery

In [15]:
kpis_df = pd.DataFrame(kpis)

print('=' * 75)
print('RESUMEN EJECUTIVO — Chicago Crimes KPIs COVID-19 (Cloud — GCS)')
print('=' * 75)
for _, row in kpis_df.iterrows():
    print(f"KPI {int(row['kpi_id']):02d} [{row['framework']:<5}] {row['kpi_name']:<45} → {row['value_text']}")
print('=' * 75)

to_bigquery(kpis_df, 'kpis_summary')

spark.stop()
print('\nEntorno cerrado. Todos los KPIs guardados en BigQuery.')

RESUMEN EJECUTIVO — Chicago Crimes KPIs COVID-19 (Cloud — GCS)
KPI 01 [Dask ] Variación Tasa de Arresto PRE→DURANTE         → PRE:20.36% → DURANTE:13.42% → POST:13.32%
KPI 02 [Dask ] Caída de Criminalidad en Confinamiento 2020   → -18.75% (2019→2020)
KPI 03 [Modin] Aumento de Violencia Doméstica DURANTE COVID  → PRE:19.20% → DURANTE:21.02% → POST:18.39%
KPI 04 [Modin] Tipo con Mayor Cambio de Resolución por COVID → PUBLIC PEACE VIOLATION: -27.88 pp (PRE→DURANTE)
KPI 05 [Modin] Distrito con Mayor Caída de Crimen en COVID   → Distrito 1: -33.89% (DURANTE vs PRE anualizado)
KPI 06 [Dask ] Recuperación Post-COVID vs Nivel PRE          → -23.54% (POST vs PRE anualizado)
KPI 07 [Dask ] Cambio en Índice de Criminalidad Nocturna     → PRE:24.42% → DURANTE:28.01% → POST:27.99%
KPI 08 [Spark] Variación Crímenes Violentos FBI Part I por COVID → PRE:15.19% → DURANTE:17.71% → POST:20.06%
KPI 09 [Spark] Cambio en Diversidad de Crímenes (Shannon) por COVID → PRE:3.4469 → DURANTE:3.4674 → POST:3.4228 

  → BigQuery: my-first-project-492901.chicago_crimes_results.kpis_summary  (10 filas)



Entorno cerrado. Todos los KPIs guardados en BigQuery.
